# Saxo Institutional Dashboard (Presentation)

Production-safe, read-only institutional monitoring notebook.
Data is sourced from live dashboard tables and runtime signal/trade history only.


In [ ]:
%pip install -q joblib

import json
from datetime import timedelta

import joblib
import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 250)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

try:
    display
except NameError:
    def display(obj):
        print(obj)

CATALOG = 'pawan_quant_research_1'
SCHEMA = 'v4'

FEED_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_feed'
HEALTH_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_health_rollup'
PERF_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_performance_rollup'
REGIME_SIGNAL_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_regime_signal'
SILVER_RT_TABLE = f'{CATALOG}.{SCHEMA}.silver_rt_1h'
BT_MACRO_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_macro_events'
LIVE_MACRO_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_macro_events_live'
BT_VIX3M_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_vix3m_1h'
BT_VIX_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_vix_1h'
TRADE_LOG_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_saxo_trade_log'
MODEL_REGISTRY_TABLE = f'{CATALOG}.{SCHEMA}.model_registry_current'

LIVE_MODEL_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/market_data/models/bt_fred_1d_xgboost_regime_live.joblib'
LOOKBACK_HOURS = 360
TOP_FEATURES = 8
FEATURE_PLOT_DAYS = 30
RECENT_TRACKING_FEATURES = ['VIX_Inversion_Z', 'VIXCLS_z', 'vix3m_z', 'PAYEMS_z', 'UNRATE_z', 'Price_Mom_24H']
RAW_FEATURE_SPECS = {
    'CPIAUCSL_z': {'type': 'macro', 'indicator_id': 'CPIAUCSL'},
    'PCEPI_z': {'type': 'macro', 'indicator_id': 'PCEPI'},
    'PCE_z': {'type': 'macro', 'indicator_id': 'PCE'},
    'PAYEMS_z': {'type': 'macro', 'indicator_id': 'PAYEMS'},
    'UNRATE_z': {'type': 'macro', 'indicator_id': 'UNRATE'},
    'GDPC1_z': {'type': 'macro', 'indicator_id': 'GDPC1'},
    'INDPRO_z': {'type': 'macro', 'indicator_id': 'INDPRO'},
    'UMCSENT_z': {'type': 'macro', 'indicator_id': 'UMCSENT'},
    'vix3m_z': {'type': 'table', 'snapshot_key': 'vix3m'},
    'VIXCLS_z': {'type': 'table', 'snapshot_key': 'vix_spot'},
    'VIX_Inversion_Z': {'type': 'derived', 'note': 'Derived feature; raw field not directly stored'},
    'Price_Mom_24H': {'type': 'derived', 'note': 'Derived feature; raw field not directly stored'},
    'CPI_Event_Active': {'type': 'derived', 'note': 'Derived event flag; raw field not directly stored'},
    'NFP_Event_Active': {'type': 'derived', 'note': 'Derived event flag; raw field not directly stored'},
}

FEATURES = [
    'CPIAUCSL_z',
    'PCEPI_z',
    'PCE_z',
    'PAYEMS_z',
    'UNRATE_z',
    'GDPC1_z',
    'INDPRO_z',
    'UMCSENT_z',
    'VIXCLS_z',
    'vix3m_z',
    'VIX_Inversion_Z',
    'Price_Mom_24H',
    'CPI_Event_Active',
    'NFP_Event_Active',
]

FEATURE_LABELS = {
    'CPIAUCSL_z': 'CPI',
    'PCEPI_z': 'Core PCE',
    'PCE_z': 'PCE',
    'PAYEMS_z': 'Payrolls',
    'UNRATE_z': 'Unemployment',
    'GDPC1_z': 'GDP',
    'INDPRO_z': 'Industrial Production',
    'UMCSENT_z': 'Sentiment',
    'VIXCLS_z': 'VIX Spot',
    'vix3m_z': 'VIX 3M',
    'VIX_Inversion_Z': 'VIX Inversion',
    'Price_Mom_24H': '24H Price Momentum',
    'CPI_Event_Active': 'CPI Event',
    'NFP_Event_Active': 'NFP Event',
}

REGIME_COLOR = {'Expansion': '#2E8B57', 'Slowdown': '#D4A017', 'Recession': '#C0392B'}
REGIME_NAME_MAP = {0: 'Expansion', 1: 'Slowdown', 2: 'Recession'}
PROBABILITY_COLUMNS = {0: 'prob_expansion', 1: 'prob_slowdown', 2: 'prob_recession'}
PROBABILITY_LABELS = {'prob_expansion': 'Expansion', 'prob_slowdown': 'Slowdown', 'prob_recession': 'Recession'}
HISTORICAL_MIX_COLUMNS = {'pct_expansion': 'Expansion', 'pct_slowdown': 'Slowdown', 'pct_recession': 'Recession'}


def _to_float(value):
    if value is None:
        return np.nan
    if isinstance(value, pd.Series):
        value = value.dropna()
        return float(value.iloc[0]) if not value.empty else np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value) if value == value else np.nan
    try:
        parsed = float(value)
        return parsed if parsed == parsed else np.nan
    except Exception:
        return np.nan


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
    except Exception:
        return False
    if isinstance(missing, (bool, int, float)):
        return bool(missing)
    return False


def _as_text(value, default='N/A'):
    return default if _is_missing(value) else str(value)


def fmt_money(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'${value:,.2f}'


def fmt_pct(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:+,.2%}'


def fmt_percent_points(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:.1f}%'


def fmt_optional_currency(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    return f'{value:,.2f}'


def fmt_optional_percent(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    return f'{value:+.2%}'


def fmt_signed_money(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    sign = '+' if value >= 0 else '-'
    return f'{sign}${abs(value):,.2f}'


def fmt_flag(value):
    if _is_missing(value):
        return 'N/A'
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {'true', 'yes', 'y', '1'}:
            return 'Yes'
        if normalized in {'false', 'no', 'n', '0'}:
            return 'No'
    return 'Yes' if bool(value) else 'No'


def fmt_raw_snapshot_value(value):
    if _is_missing(value):
        return 'N/A'
    if isinstance(value, str):
        return value
    numeric = _to_float(value)
    if pd.notna(numeric):
        if numeric in (0.0, 1.0):
            return fmt_flag(int(numeric))
        return f'{numeric:,.4f}'
    return str(value)


def fmt_minutes(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:.0f}'


def fmt_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    return 'N/A' if pd.isna(ts) else ts.strftime('%Y-%m-%d %H:%M:%S')


def table_exists(name):
    return spark.catalog.tableExists(name)


def query_to_pandas(sql, empty_columns=None):
    try:
        return spark.sql(sql).toPandas()
    except Exception as exc:
        print(f'Query failed: {exc}')
        return pd.DataFrame(columns=empty_columns or [])


def latest_row(table_name, order_col):
    if not table_exists(table_name):
        return pd.Series(dtype='object')
    rows = query_to_pandas(f'SELECT * FROM {table_name} ORDER BY {order_col} DESC LIMIT 1')
    return rows.iloc[0] if not rows.empty else pd.Series(dtype='object')


def latest_rows(table_name, order_col, limit=LOOKBACK_HOURS):
    if not table_exists(table_name):
        return pd.DataFrame()
    q = f'''
        SELECT *
        FROM (
            SELECT *
            FROM {table_name}
            ORDER BY {order_col} DESC
            LIMIT {int(limit)}
        ) t
        ORDER BY {order_col} ASC
    '''
    return query_to_pandas(q)


def table_columns(table_name):
    schema = query_to_pandas(f'DESCRIBE {table_name}', empty_columns=['col_name'])
    if schema.empty or 'col_name' not in schema.columns:
        return []
    return [str(c).strip() for c in schema['col_name'].tolist()]


def load_model_metadata():
    default_version = 'bootstrap_live_model'
    default_path = LIVE_MODEL_PATH
    if table_exists(MODEL_REGISTRY_TABLE):
        rows = query_to_pandas(
            f'''
                SELECT model_version, model_path
                FROM {MODEL_REGISTRY_TABLE}
                ORDER BY promoted_ts DESC
                LIMIT 1
            '''
        )
        if not rows.empty and not _is_missing(rows.iloc[0].get('model_path')):
            return _as_text(rows.iloc[0].get('model_version'), default_version), rows.iloc[0].get('model_path')
    return default_version, default_path


def load_model():
    model_version, model_path = load_model_metadata()
    try:
        model = joblib.load(model_path)
        return model, model_version, model_path
    except Exception as exc:
        print(f'Model load skipped ({model_path}): {exc}')
        return None, model_version, model_path


def feature_importance(model):
    if model is not None and hasattr(model, 'feature_importances_'):
        importance = np.array(model.feature_importances_, dtype=float)
    else:
        importance = np.ones(len(FEATURES), dtype=float)

    if importance.ndim != 1 or importance.size != len(FEATURES) or np.isclose(np.nan_to_num(importance).sum(), 0):
        importance = np.ones(len(FEATURES), dtype=float)

    frame = pd.DataFrame({'feature': FEATURES, 'importance': importance})
    frame = frame.sort_values('importance', ascending=False).reset_index(drop=True)
    frame['importance_pct'] = frame['importance'] / frame['importance'].sum()
    frame['label'] = frame['feature'].map(FEATURE_LABELS)
    return frame


def top_named_value_from_row(row, columns):
    best_label = 'N/A'
    best_value = np.nan
    for c, label in columns.items():
        val = _to_float(row.get(c))
        if pd.notna(val) and (pd.isna(best_value) or val > best_value):
            best_label = label
            best_value = val
    return best_label, best_value


def historical_mix_text(row):
    mix_parts = []
    for column, label in HISTORICAL_MIX_COLUMNS.items():
        value = _to_float(row.get(column))
        if pd.notna(value):
            mix_parts.append(f'{label}: {fmt_percent_points(value)}')
    return ' | '.join(mix_parts) if mix_parts else 'N/A'


def compute_probabilities(frame, model):
    probability_df = pd.DataFrame(index=frame.index)
    for column in PROBABILITY_COLUMNS.values():
        probability_df[column] = np.nan

    if frame.empty or model is None or not hasattr(model, 'predict_proba'):
        return probability_df

    feature_input = frame.reindex(columns=FEATURES).apply(pd.to_numeric, errors='coerce').fillna(0.0)
    try:
        probs = model.predict_proba(feature_input)
        classes = list(getattr(model, 'classes_', [0, 1, 2]))
        for idx, class_id in enumerate(classes):
            try:
                column_name = PROBABILITY_COLUMNS.get(int(class_id))
            except Exception:
                column_name = None
            if column_name and idx < probs.shape[1]:
                probability_df[column_name] = probs[:, idx]
    except Exception as exc:
        print(f'Probability computation unavailable: {exc}')

    return probability_df


def detect_table_columns(frame, candidates):
    cols = set(frame.columns) if hasattr(frame, 'columns') else set(frame)
    return [c for c in candidates if c in cols]


def regime_label(name, code):
    text = _as_text(name, '')
    if text:
        return text
    code_value = _to_float(code)
    if pd.isna(code_value):
        return 'Unknown'
    return REGIME_NAME_MAP.get(int(code_value), f'Regime {int(code_value)}')


def build_regime_intervals(signal_frame):
    if signal_frame.empty or 'timestamp' not in signal_frame.columns:
        return pd.DataFrame(columns=['start_ts', 'end_ts', 'regime_name', 'regime_int', 'position'])

    frame = signal_frame.copy()
    frame = frame.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    if frame.empty:
        return pd.DataFrame(columns=['start_ts', 'end_ts', 'regime_name', 'regime_int', 'position'])

    name_series = frame['regime_name'] if 'regime_name' in frame.columns else pd.Series(index=frame.index, dtype='object')
    code_series = frame['regime_int'] if 'regime_int' in frame.columns else pd.Series(index=frame.index, dtype='float64')
    frame['regime_name'] = [regime_label(name, code) for name, code in zip(name_series, code_series)]

    change_mask = frame['regime_name'].ne(frame['regime_name'].shift())
    if 'regime_int' in frame.columns:
        regime_codes = pd.to_numeric(frame['regime_int'], errors='coerce')
        change_mask = change_mask | regime_codes.ne(regime_codes.shift())
    if 'position' in frame.columns:
        positions = frame['position'].fillna('')
        change_mask = change_mask | positions.ne(positions.shift().fillna(''))

    interval_cols = ['timestamp', 'regime_name'] + detect_table_columns(frame, ['regime_int', 'position'])
    intervals = frame.loc[change_mask, interval_cols].copy()
    intervals = intervals.rename(columns={'timestamp': 'start_ts'})
    intervals['end_ts'] = intervals['start_ts'].shift(-1)
    return intervals.reset_index(drop=True)


def load_latest_macro_snapshots(indicator_ids):
    snapshots = []
    if not indicator_ids:
        return pd.DataFrame(columns=['indicator_id', 'raw_value', 'raw_ts', 'source_table', 'source_field'])
    indicators_sql = ', '.join([f"'{indicator}'" for indicator in indicator_ids])

    if table_exists(LIVE_MACRO_TABLE):
        live_cols = set(table_columns(LIVE_MACRO_TABLE))
        if {'indicator_id', 'observed_value', 'scheduled_release_ts'}.issubset(live_cols):
            live_ts = 'COALESCE(observed_timestamp, scheduled_release_ts)' if 'observed_timestamp' in live_cols else 'scheduled_release_ts'
            live_sql = f"""
                SELECT indicator_id, observed_value AS raw_value, raw_ts, source_table, source_field
                FROM (
                    SELECT indicator_id,
                           observed_value,
                           {live_ts} AS raw_ts,
                           '{LIVE_MACRO_TABLE}' AS source_table,
                           'observed_value' AS source_field,
                           ROW_NUMBER() OVER (PARTITION BY indicator_id ORDER BY {live_ts} DESC, ingestion_ts DESC) AS rn
                    FROM {LIVE_MACRO_TABLE}
                    WHERE indicator_id IN ({indicators_sql})
                      AND observed_value IS NOT NULL
                ) t
                WHERE rn = 1
            """
            snapshots.append(query_to_pandas(live_sql))

    if table_exists(BT_MACRO_TABLE):
        bt_cols = set(table_columns(BT_MACRO_TABLE))
        value_field = 'actual' if 'actual' in bt_cols else 'observed_value' if 'observed_value' in bt_cols else None
        ts_field = 'timestamp' if 'timestamp' in bt_cols else 'scheduled_release_ts' if 'scheduled_release_ts' in bt_cols else None
        if value_field and ts_field and 'indicator_id' in bt_cols:
            bt_sql = f"""
                SELECT indicator_id, {value_field} AS raw_value, {ts_field} AS raw_ts, '{BT_MACRO_TABLE}' AS source_table, '{value_field}' AS source_field
                FROM (
                    SELECT indicator_id,
                           {value_field},
                           {ts_field},
                           ROW_NUMBER() OVER (PARTITION BY indicator_id ORDER BY {ts_field} DESC) AS rn
                    FROM {BT_MACRO_TABLE}
                    WHERE indicator_id IN ({indicators_sql})
                      AND {value_field} IS NOT NULL
                ) t
                WHERE rn = 1
            """
            snapshots.append(query_to_pandas(bt_sql))

    if not snapshots:
        return pd.DataFrame(columns=['indicator_id', 'raw_value', 'raw_ts', 'source_table', 'source_field'])

    macro_snapshots = pd.concat(snapshots, ignore_index=True)
    macro_snapshots['raw_ts'] = pd.to_datetime(macro_snapshots['raw_ts'], errors='coerce')
    macro_snapshots = macro_snapshots.sort_values(['indicator_id', 'raw_ts'], ascending=[True, False]).drop_duplicates(subset=['indicator_id'], keep='first')
    return macro_snapshots.reset_index(drop=True)


def load_latest_table_snapshot(table_name, value_candidates, ts_field='timestamp'):
    if not table_exists(table_name):
        return None
    cols = set(table_columns(table_name))
    value_field = next((candidate for candidate in value_candidates if candidate in cols), None)
    if value_field is None or ts_field not in cols:
        return None
    row = latest_row(table_name, ts_field)
    if row.empty:
        return None
    return {
        'raw_value': row.get(value_field),
        'raw_ts': row.get(ts_field),
        'source_table': table_name,
        'source_field': value_field,
    }


def resolve_raw_feature_snapshot(feature, macro_snapshots, table_snapshots):
    spec = RAW_FEATURE_SPECS.get(feature, {})
    feature_type = spec.get('type')

    if feature_type == 'macro':
        indicator_id = spec.get('indicator_id')
        match = macro_snapshots[macro_snapshots['indicator_id'] == indicator_id] if not macro_snapshots.empty else pd.DataFrame()
        if not match.empty:
            row = match.iloc[0]
            return {
                'Raw value': fmt_raw_snapshot_value(row.get('raw_value')),
                'Latest timestamp': fmt_timestamp(row.get('raw_ts')),
                'Source table': _as_text(row.get('source_table')),
                'Source field': _as_text(row.get('source_field')),
                'Availability': 'Latest macro observation',
            }
        return {
            'Raw value': 'N/A',
            'Latest timestamp': 'N/A',
            'Source table': BT_MACRO_TABLE,
            'Source field': 'actual / observed_value',
            'Availability': 'No raw macro observation available',
        }

    if feature_type == 'table':
        snapshot = table_snapshots.get(spec.get('snapshot_key'))
        if snapshot and not _is_missing(snapshot.get('raw_value')):
            return {
                'Raw value': fmt_raw_snapshot_value(snapshot.get('raw_value')),
                'Latest timestamp': fmt_timestamp(snapshot.get('raw_ts')),
                'Source table': _as_text(snapshot.get('source_table')),
                'Source field': _as_text(snapshot.get('source_field')),
                'Availability': 'Latest source-table reading',
            }
        fallback_table = {'vix3m': BT_VIX3M_TABLE, 'vix_spot': BT_VIX_TABLE}.get(spec.get('snapshot_key'), 'N/A')
        return {
            'Raw value': 'N/A',
            'Latest timestamp': 'N/A',
            'Source table': fallback_table,
            'Source field': 'close',
            'Availability': 'No reliable latest source value available',
        }

    return {
        'Raw value': 'N/A',
        'Latest timestamp': 'N/A',
        'Source table': 'N/A',
        'Source field': 'N/A',
        'Availability': spec.get('note', 'Raw field not directly stored'),
    }


latest_feed = latest_row(FEED_TABLE, 'feed_ts')
latest_health = latest_row(HEALTH_TABLE, 'rollup_ts')
latest_perf = latest_row(PERF_TABLE, 'rollup_ts')
execution_mode = _as_text(latest_feed.get('execution_mode', ''), '') if hasattr(latest_feed, 'get') else ''
if not execution_mode:
    execution_mode = _as_text(latest_perf.get('execution_mode', ''), '')
safe_execution_mode = execution_mode.replace("'", "''") if execution_mode else ''
mode_filter = '' if not safe_execution_mode else f"WHERE execution_mode = '{safe_execution_mode}'"

perf_history = query_to_pandas(f"""
    SELECT *
    FROM {PERF_TABLE}
    {mode_filter}
    ORDER BY rollup_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

health_history = query_to_pandas(f"""
    SELECT *
    FROM {HEALTH_TABLE}
    {mode_filter}
    ORDER BY rollup_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

signal_history = query_to_pandas(f"""
    SELECT *
    FROM (
      SELECT timestamp, regime_name, CAST(regime_int AS INT) AS regime_int, position
      FROM {REGIME_SIGNAL_TABLE}
      ORDER BY timestamp DESC
      LIMIT {LOOKBACK_HOURS}
    ) t
    ORDER BY timestamp ASC
""")

silver_cols = table_columns(SILVER_RT_TABLE)
if silver_cols:
    requested = [f for f in FEATURES if f in silver_cols]
    query_cols = ', '.join(['timestamp'] + requested)
    feature_history = query_to_pandas(f"SELECT {query_cols} FROM {SILVER_RT_TABLE} ORDER BY timestamp DESC LIMIT {LOOKBACK_HOURS}")
else:
    feature_history = pd.DataFrame(columns=['timestamp'] + FEATURES)
macro_indicator_ids = [spec['indicator_id'] for spec in RAW_FEATURE_SPECS.values() if spec.get('type') == 'macro']
macro_raw_snapshots = load_latest_macro_snapshots(sorted(set(macro_indicator_ids)))
table_raw_snapshots = {
    'vix3m': load_latest_table_snapshot(BT_VIX3M_TABLE, ['close', 'vix3m']),
    'vix_spot': load_latest_table_snapshot(BT_VIX_TABLE, ['close']),
}

if 'timestamp' in feature_history.columns:
    feature_history['timestamp'] = pd.to_datetime(feature_history['timestamp'], errors='coerce')
    feature_history = feature_history.sort_values('timestamp').reset_index(drop=True)
    for col in [c for c in feature_history.columns if c != 'timestamp']:
        feature_history[col] = pd.to_numeric(feature_history[col], errors='coerce')

trade_history = query_to_pandas(f"SELECT * FROM {TRADE_LOG_TABLE} ORDER BY trade_ts DESC LIMIT 250", empty_columns=['trade_ts'])
if 'trade_ts' in trade_history.columns:
    trade_history['trade_ts'] = pd.to_datetime(trade_history['trade_ts'], errors='coerce')
    trade_history = trade_history.sort_values('trade_ts').reset_index(drop=True)

model, model_version, model_path = load_model()
importance_frame = feature_importance(model)
importance_value_map = dict(zip(importance_frame['feature'], importance_frame['importance']))
importance_pct_map = dict(zip(importance_frame['feature'], importance_frame['importance_pct']))
probability_history = pd.DataFrame(columns=['timestamp'] + list(PROBABILITY_LABELS.keys()))
if not feature_history.empty and 'timestamp' in feature_history.columns:
    probability_history = pd.concat(
        [
            feature_history[['timestamp']].reset_index(drop=True),
            compute_probabilities(feature_history, model).reset_index(drop=True),
        ],
        axis=1,
    )

perf_history = perf_history.sort_values('rollup_ts').reset_index(drop=True) if not perf_history.empty else perf_history
if not perf_history.empty:
    perf_history['rollup_ts'] = pd.to_datetime(perf_history['rollup_ts'], errors='coerce')
    for c in ['account_equity', 'account_cash', 'drawdown_pct', 'regime_drawdown_pct', 'alpha_pct', 'strategy_return_1h', 'alpha_1h', 'strategy_return_1d', 'alpha_1d']:
        if c in perf_history.columns:
            perf_history[c] = pd.to_numeric(perf_history[c], errors='coerce')

if not health_history.empty:
    health_history['rollup_ts'] = pd.to_datetime(health_history['rollup_ts'], errors='coerce')
    health_history = health_history.sort_values('rollup_ts').reset_index(drop=True)

if not signal_history.empty:
    signal_history['timestamp'] = pd.to_datetime(signal_history['timestamp'], errors='coerce')
    signal_history = signal_history.sort_values('timestamp').reset_index(drop=True)
    signal_history['regime_int'] = pd.to_numeric(signal_history['regime_int'], errors='coerce')
    signal_history['regime_name'] = [regime_label(name, code) for name, code in zip(signal_history.get('regime_name', pd.Series(index=signal_history.index, dtype='object')), signal_history.get('regime_int', pd.Series(index=signal_history.index, dtype='float64')))]

regime_history = signal_history.copy()
regime_intervals = build_regime_intervals(signal_history)
latest_probability_rows = probability_history.dropna(subset=list(PROBABILITY_LABELS.keys()), how='all') if not probability_history.empty else pd.DataFrame()
latest_probability = latest_probability_rows.iloc[-1] if not latest_probability_rows.empty else pd.Series(dtype='float64')


displayHTML('<h2>1. Executive decision header</h2>')
prob_label, prob_value = top_named_value_from_row(latest_probability, PROBABILITY_LABELS)
kpi = [
    ('Current regime', _as_text(latest_feed.get('regime_name', 'N/A'))),
    ('Current model probability', f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else 'N/A'),
    ('Historical regime mix', historical_mix_text(latest_feed)),
    ('Current position', _as_text(latest_feed.get('actual_ticker', latest_feed.get('position', 'N/A')))),
    ('Execution mode', _as_text(latest_feed.get('execution_mode', latest_perf.get('execution_mode', 'N/A')))),
    ('Account equity', fmt_money(latest_feed.get('account_equity'))),
    ('Account cash', fmt_money(latest_feed.get('account_cash'))),
    ('Account balance', fmt_money(latest_feed.get('account_balance'))),
    ('1h strategy', fmt_pct(latest_perf.get('strategy_return_1h'))),
    ('1h alpha', fmt_pct(latest_perf.get('alpha_1h'))),
    ('1d strategy', fmt_pct(latest_perf.get('strategy_return_1d'))),
    ('1d alpha', fmt_pct(latest_perf.get('alpha_1d'))),
    ('Current drawdown', fmt_pct(latest_feed.get('drawdown_pct'))),
    ('Freshness (min)', fmt_minutes(latest_feed.get('dashboard_data_freshness_minutes'))),
    ('Latest audit timestamp', fmt_timestamp(latest_feed.get('latest_audit_ts'))),
    ('Last execution action', _as_text(latest_feed.get('last_execution_action'))),
    ('Skip reason', _as_text(latest_feed.get('last_skip_reason'))),
    ('Error reason', _as_text(latest_feed.get('last_error_reason'))),
]
display(pd.DataFrame(kpi, columns=['Metric', 'Value']))


displayHTML('<h2>2. Strategy vs SPY performance</h2>')
if perf_history.empty:
    print('No strategy performance history is available yet.')
else:
    aligned = perf_history.copy()
    aligned = aligned.dropna(subset=['rollup_ts', 'account_equity'])
    if aligned.empty:
        print('No aligned performance records available.')
    else:
        aligned['strategy_return'] = aligned['account_equity'] / aligned['account_equity'].iloc[0] - 1.0
        if 'account_cash' in aligned.columns and aligned['account_cash'].notna().all():
            aligned['cash_return'] = aligned['account_cash'] / aligned['account_cash'].iloc[0] - 1.0

        spy_df = pd.DataFrame()
        if 'spy_close' in trade_history.columns:
            spy_df = trade_history[['trade_ts', 'spy_close']].copy() if 'trade_ts' in trade_history.columns else pd.DataFrame()
            spy_df = spy_df.dropna()

        if not spy_df.empty:
            spy_df = spy_df.sort_values('trade_ts')
            aligned = pd.merge_asof(
                aligned.sort_values('rollup_ts'),
                spy_df.sort_values('trade_ts'),
                left_on='rollup_ts',
                right_on='trade_ts',
                direction='backward'
            )
            aligned['spy_close'] = pd.to_numeric(aligned['spy_close'], errors='coerce')
            if aligned['spy_close'].notna().any():
                first_spy = float(aligned['spy_close'].dropna().iloc[0])
                aligned['spy_return'] = aligned['spy_close'] / first_spy - 1.0

        fig = plt.figure(figsize=(16, 10), constrained_layout=True)
        gs = fig.add_gridspec(3, 1, height_ratios=[3.2, 1.2, 1.0])
        ax_perf = fig.add_subplot(gs[0, 0])
        ax_alpha = fig.add_subplot(gs[1, 0], sharex=ax_perf)
        ax_dd = fig.add_subplot(gs[2, 0], sharex=ax_perf)

        if not regime_intervals.empty:
            window_start = aligned['rollup_ts'].iloc[0]
            window_end = aligned['rollup_ts'].iloc[-1]
            for idx, row in regime_intervals.iterrows():
                start_t = row['start_ts']
                end_t = row['end_ts'] if pd.notna(row.get('end_ts')) else window_end
                if pd.isna(start_t) or pd.isna(end_t):
                    continue
                if idx == 0 and start_t > window_start:
                    start_t = window_start
                if start_t < window_start:
                    start_t = window_start
                if start_t >= window_end or end_t <= start_t:
                    continue
                color = REGIME_COLOR.get(_as_text(row['regime_name'], 'Unknown'), '#9E9E9E')
                ax_perf.axvspan(start_t, min(end_t, window_end), color=color, alpha=0.15)

        ax_perf.plot(aligned['rollup_ts'], aligned['strategy_return'] * 100, linewidth=2.2, label='Strategy')
        if 'spy_return' in aligned.columns:
            ax_perf.plot(aligned['rollup_ts'], aligned['spy_return'] * 100, linewidth=1.5, linestyle='--', label='SPY proxy', alpha=0.8)
        ax_perf.set_title('Saxo strategy vs SPY')
        ax_perf.set_ylabel('Return (%)')
        ax_perf.grid(alpha=0.3)
        ax_perf.legend(loc='upper left')

        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_pct', np.nan), errors='coerce') * 100, linewidth=2.0, label='Alpha')
        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_1h', np.nan), errors='coerce') * 100, linewidth=1.3, linestyle=':', label='Alpha 1h')
        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_1d', np.nan), errors='coerce') * 100, linewidth=1.3, linestyle='--', label='Alpha 1d')
        ax_alpha.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax_alpha.set_ylabel('Alpha (%)')
        ax_alpha.set_title('Alpha panel')
        ax_alpha.grid(alpha=0.3)
        ax_alpha.legend(loc='upper left')

        drawdown_series = pd.to_numeric(aligned.get('drawdown_pct', np.nan), errors='coerce').fillna(0)
        ax_dd.plot(aligned['rollup_ts'], drawdown_series * 100, linewidth=1.8, color='#B03A2E', label='Strategy drawdown')
        if 'regime_drawdown_pct' in aligned.columns:
            ax_dd.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('regime_drawdown_pct', np.nan), errors='coerce') * 100, linewidth=1.3, color='#D35400', label='Regime drawdown')
        ax_dd.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax_dd.set_ylabel('Drawdown (%)')
        ax_dd.set_title('Drawdown panel')
        ax_dd.grid(alpha=0.3)
        ax_dd.legend(loc='lower right')
        plt.show()


displayHTML('<h2>3. Historical regime / signal monitoring</h2>')
if not regime_history.empty:
    displayHTML('<p><i>Model probabilities below are computed on the fly from the current promoted model using recent feature history. Hourly regime history remains sourced from bronze_rt_regime_signal.</i></p>')
    regime_head_cols = [c for c in ['timestamp', 'regime_name', 'regime_int', 'position'] if c in regime_history.columns]
    regime_head = regime_history[regime_head_cols].copy()
    regime_head['timestamp'] = regime_head['timestamp'].apply(fmt_timestamp)
    regime_head = regime_head.sort_values('timestamp', ascending=False).head(24)
    regime_head = regime_head.rename(columns={'timestamp': 'Timestamp', 'regime_name': 'Regime', 'regime_int': 'Regime code', 'position': 'Target position'})
    display(regime_head)

    fig2, (axp, axr) = plt.subplots(2, 1, figsize=(16, 7.2), sharex=True, gridspec_kw={'height_ratios': [2.1, 1.0]}, constrained_layout=True)
    probability_plot_columns = [column for column in PROBABILITY_LABELS.keys() if column in probability_history.columns and probability_history[column].notna().any()]
    if probability_plot_columns:
        for column in probability_plot_columns:
            axp.step(probability_history['timestamp'], probability_history[column] * 100, where='post', linewidth=1.7, label=PROBABILITY_LABELS[column])
        axp.set_title('Historical model probabilities (computed on the fly)')
        axp.set_ylabel('Probability (%)')
        axp.set_ylim(0, 100)
        axp.grid(alpha=0.25)
        axp.legend(loc='upper left', ncol=3)
    else:
        axp.axis('off')
        axp.text(0.5, 0.5, 'Model probability history is unavailable because predict_proba could not be computed.', ha='center', va='center')

    sig_plot = regime_history[['timestamp', 'regime_int']].dropna().copy()
    if not sig_plot.empty:
        sig_plot['regime_step'] = pd.to_numeric(sig_plot['regime_int'], errors='coerce')
        axr.step(sig_plot['timestamp'], sig_plot['regime_step'], where='post', linewidth=2.0, color='#1F4E79')
        axr.set_ylim(-0.1, 2.1)
        axr.set_ylabel('Regime')
        axr.set_title('Hourly regime history')
        axr.set_yticks([0, 1, 2])
        axr.set_yticklabels(['Expansion', 'Slowdown', 'Recession'])
        axr.grid(alpha=0.25)
    else:
        axr.axis('off')
        axr.text(0.5, 0.5, 'No signal history available.', ha='center', va='center')

    plt.show()
else:
    print('No regime signal history available from bronze_rt_regime_signal.')


displayHTML('<h2>4. Current feature state</h2>')
current_features = pd.DataFrame(columns=['feature', 'label', 'model_importance', 'importance_pct', 'current_z_score', 'delta_vs_prior', 'weighted_move_proxy'])
if feature_history.empty:
    print('No feature history available in silver.')
else:
    latest = feature_history.tail(1).iloc[0]
    prev = feature_history.tail(2).iloc[0] if len(feature_history) > 1 else pd.Series(dtype=float)
    rows = []
    for feature in FEATURES:
        if feature not in feature_history.columns:
            continue
        cur = _to_float(latest.get(feature))
        p = _to_float(prev.get(feature)) if not prev.empty else np.nan
        delta_z = cur - p if not pd.isna(cur) and not pd.isna(p) else np.nan
        model_importance = _to_float(importance_value_map.get(feature, np.nan))
        importance_pct = _to_float(importance_pct_map.get(feature, np.nan))
        rows.append((
            feature,
            FEATURE_LABELS.get(feature, feature),
            model_importance,
            importance_pct,
            _to_float(cur),
            delta_z,
            abs(importance_pct * delta_z) if not pd.isna(delta_z) and not pd.isna(importance_pct) else np.nan,
        ))

    current_features = pd.DataFrame(rows, columns=['feature', 'label', 'model_importance', 'importance_pct', 'current_z_score', 'delta_vs_prior', 'weighted_move_proxy'])
    current_features = current_features.sort_values('model_importance', ascending=False).reset_index(drop=True)
    displayHTML('<p><i>First view ranks the largest recent weighted movers; second view ranks the highest model-importance features.</i></p>')
    feature_display_columns = {'label': 'Feature', 'model_importance': 'Model importance', 'importance_pct': 'Importance %', 'current_z_score': 'Current z-score', 'delta_vs_prior': 'Delta vs previous bar', 'weighted_move_proxy': 'Weighted move proxy'}
    weighted_movers_display = current_features.sort_values('weighted_move_proxy', ascending=False).head(TOP_FEATURES).drop(columns=['feature']).rename(columns=feature_display_columns)
    importance_leaders_display = current_features.sort_values('model_importance', ascending=False).head(TOP_FEATURES).drop(columns=['feature']).rename(columns=feature_display_columns)
    displayHTML('<h4>Largest recent weighted movers</h4>')
    display(weighted_movers_display.style.format({'Model importance': '{:.4f}', 'Importance %': '{:.2%}', 'Current z-score': '{:+.3f}', 'Delta vs previous bar': '{:+.3f}', 'Weighted move proxy': '{:.4f}'}, na_rep='N/A'))
    displayHTML('<h4>Highest model importance</h4>')
    display(importance_leaders_display.style.format({'Model importance': '{:.4f}', 'Importance %': '{:.2%}', 'Current z-score': '{:+.3f}', 'Delta vs previous bar': '{:+.3f}', 'Weighted move proxy': '{:.4f}'}, na_rep='N/A'))
    displayHTML('<h4>Current raw feature snapshot</h4>')
    displayHTML('<p><i>Z-score is the normalized model input. Raw value is the latest underlying source reading where a real field is available.</i></p>')
    raw_snapshot_rows = []
    for feature in current_features.sort_values('model_importance', ascending=False).head(TOP_FEATURES)['feature'].tolist():
        raw_details = resolve_raw_feature_snapshot(feature, macro_raw_snapshots, table_raw_snapshots)
        raw_snapshot_rows.append({'Feature': FEATURE_LABELS.get(feature, feature), **raw_details})
    raw_snapshot_df = pd.DataFrame(raw_snapshot_rows)
    display(raw_snapshot_df)


displayHTML('<h2>5. Historical feature monitoring</h2>')
if feature_history.empty or current_features.empty:
    print('No feature monitoring data available.')
else:
    displayHTML('<p><i>Recent feature tracking combines the most decision-relevant series, while 24H momentum stays on a dedicated right axis for visibility.</i></p>')
    recent_feature_window = feature_history.tail(24 * FEATURE_PLOT_DAYS).copy()
    combined_features = [feature for feature in RECENT_TRACKING_FEATURES if feature in recent_feature_window.columns and feature != 'Price_Mom_24H']
    if not recent_feature_window.empty and combined_features:
        fig_combo, ax_combo = plt.subplots(figsize=(16, 5.2), constrained_layout=True)
        combo_colors = {'VIX_Inversion_Z': '#1F4E79', 'VIXCLS_z': '#B03A2E', 'vix3m_z': '#7D6608', 'PAYEMS_z': '#117A65', 'UNRATE_z': '#6C3483'}
        for feature in combined_features:
            ax_combo.plot(recent_feature_window['timestamp'], pd.to_numeric(recent_feature_window[feature], errors='coerce'), linewidth=1.5, label=FEATURE_LABELS.get(feature, feature), color=combo_colors.get(feature, '#1F4E79'))
        ax_combo.axhline(0, color='black', linestyle=':', alpha=0.45)
        ax_combo.set_title(f'Recent Feature Tracking ({FEATURE_PLOT_DAYS}D)')
        ax_combo.set_ylabel('Feature z-score')
        ax_combo.grid(alpha=0.25)
        momentum_axis = None
        if 'Price_Mom_24H' in recent_feature_window.columns:
            momentum_axis = ax_combo.twinx()
            momentum_axis.plot(recent_feature_window['timestamp'], pd.to_numeric(recent_feature_window['Price_Mom_24H'], errors='coerce'), linewidth=1.8, linestyle='--', color='#C0392B', label=FEATURE_LABELS.get('Price_Mom_24H', 'Price_Mom_24H'))
            momentum_axis.axhline(0, color='#C0392B', linestyle=':', alpha=0.3)
            momentum_axis.set_ylabel('24H momentum')
        combo_handles, combo_labels = ax_combo.get_legend_handles_labels()
        if momentum_axis is not None:
            momentum_handles, momentum_labels = momentum_axis.get_legend_handles_labels()
            combo_handles += momentum_handles
            combo_labels += momentum_labels
        ax_combo.legend(combo_handles, combo_labels, loc='upper left', ncol=3)
        plt.show()

    displayHTML('<p><i>Top feature history by model importance over the trailing window.</i></p>')
    panel_features = [f for f in current_features['feature'].tolist() if f in feature_history.columns and f != 'Price_Mom_24H'][:6]
    selected_window = recent_feature_window[['timestamp'] + panel_features].copy()
    if panel_features:
        n_cols = 2 if len(panel_features) > 1 else 1
        n_rows = math.ceil(len(panel_features) / n_cols)
        fig3, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.2 * n_rows), sharex=True, constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()
        for axis, feature in zip(axes, panel_features):
            series = pd.to_numeric(selected_window[feature], errors='coerce')
            axis.plot(selected_window['timestamp'], series, linewidth=1.5, color='#1F4E79')
            axis.set_ylabel('z-score')
            axis.axhline(0, color='black', linestyle=':', alpha=0.5)
            axis.set_title(FEATURE_LABELS.get(feature, feature))
            axis.grid(alpha=0.3)
        for axis in axes[len(panel_features):]:
            axis.axis('off')
        plt.show()
    else:
        print('No top feature series available in silver_rt_1h.')

    if 'Price_Mom_24H' in feature_history.columns:
        mom = feature_history[['timestamp', 'Price_Mom_24H']].dropna()
        if not mom.empty:
            mom_window = mom.tail(24 * FEATURE_PLOT_DAYS)
            fig4, ax4 = plt.subplots(figsize=(16, 3.8), constrained_layout=True)
            ax4.bar(mom_window['timestamp'], mom_window['Price_Mom_24H'], color=['#2E8B57' if v >= 0 else '#C0392B' for v in mom_window['Price_Mom_24H']], width=0.03, alpha=0.65)
            ax4.set_title('24H momentum (dedicated panel)')
            ax4.set_ylabel('z-score')
            ax4.set_xlabel('Timestamp')
            ax4.axhline(0, color='black', linestyle='--', alpha=0.5)
            ax4.grid(alpha=0.3)
            ax4.margins(x=0.01)
            plt.show()


displayHTML('<h2>6. Attribution</h2>')
displayHTML('<p><i>Top weighted movers vs previous bar. Descriptive proxy only; not causal and not a SHAP explanation.</i></p>')
if feature_history.empty or current_features.empty:
    print('No attribution history available.')
else:
    probability_context_df = pd.DataFrame(
        [
            {
                'Top probability': f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else 'N/A',
                'Expansion': fmt_pct(latest_probability.get('prob_expansion')),
                'Slowdown': fmt_pct(latest_probability.get('prob_slowdown')),
                'Recession': fmt_pct(latest_probability.get('prob_recession')),
            }
        ]
    )
    display(probability_context_df)
    bridge_display = current_features.sort_values('weighted_move_proxy', ascending=False).head(5).drop(columns=['feature']).copy()
    bridge_display = bridge_display.rename(columns={'label': 'Feature', 'model_importance': 'Model importance', 'importance_pct': 'Importance %', 'current_z_score': 'Current z-score', 'delta_vs_prior': 'Delta vs previous bar', 'weighted_move_proxy': 'Weighted move proxy'})
    display(bridge_display.style.format({'Model importance': '{:.4f}', 'Importance %': '{:.2%}', 'Current z-score': '{:+.3f}', 'Delta vs previous bar': '{:+.3f}', 'Weighted move proxy': '{:.4f}'}, na_rep='N/A'))


displayHTML('<h2>7. Historical signals and trades</h2>')
sig_cols = [
    'timestamp', 'regime_name', 'regime_int', 'position'
]
if not signal_history.empty:
    available_sig = [c for c in sig_cols if c in signal_history.columns]
    print('Latest 24 hourly signals:')
    signal_tail = signal_history[available_sig].sort_values('timestamp', ascending=False).head(24).copy()
    if 'timestamp' in signal_tail.columns:
        signal_tail['timestamp'] = signal_tail['timestamp'].apply(fmt_timestamp)
    if 'regime_name' in signal_tail.columns:
        signal_tail['regime_name'] = signal_tail['regime_name'].apply(_as_text)
    if 'position' in signal_tail.columns:
        signal_tail['position'] = signal_tail['position'].apply(_as_text)
    signal_tail = signal_tail.rename(columns={'timestamp': 'Timestamp', 'regime_name': 'Regime', 'regime_int': 'Regime code', 'position': 'Target position'})
    display(signal_tail)

if trade_history.empty:
    print('No trade history available in window.')
else:
    trade_cols = ['trade_ts', 'signal_ts', 'regime_name', 'target_position', 'switched', 'target_ticker', 'account_balance', 'account_equity', 'hourly_pnl', 'hourly_pnl_pct', 'day_pnl', 'day_pnl_pct', 'execution_mode']
    available_cols = [c for c in trade_cols if c in trade_history.columns]
    trade_tail = trade_history[available_cols].sort_values('trade_ts', ascending=False).head(100).copy()
    for ts_col in ['trade_ts', 'signal_ts']:
        if ts_col in trade_tail.columns:
            trade_tail[ts_col] = trade_tail[ts_col].apply(fmt_timestamp)
    for text_col in ['regime_name', 'target_position', 'target_ticker', 'execution_mode']:
        if text_col in trade_tail.columns:
            trade_tail[text_col] = trade_tail[text_col].apply(_as_text)
    if 'switched' in trade_tail.columns:
        trade_tail['switched'] = trade_tail['switched'].apply(fmt_flag)
    for money_col in ['account_balance', 'account_equity']:
        if money_col in trade_tail.columns:
            trade_tail[money_col] = trade_tail[money_col].apply(fmt_money)
    for pnl_col in ['hourly_pnl', 'day_pnl']:
        if pnl_col in trade_tail.columns:
            trade_tail[pnl_col] = trade_tail[pnl_col].apply(fmt_signed_money)
    for pct_col in ['hourly_pnl_pct', 'day_pnl_pct']:
        if pct_col in trade_tail.columns:
            trade_tail[pct_col] = trade_tail[pct_col].apply(fmt_optional_percent)
    trade_tail = trade_tail.rename(columns={'trade_ts': 'Trade timestamp', 'signal_ts': 'Signal timestamp', 'regime_name': 'Regime', 'target_position': 'Target position', 'switched': 'Switched', 'target_ticker': 'Target ticker', 'account_balance': 'Account balance', 'account_equity': 'Account equity', 'hourly_pnl': 'Hourly pnl', 'hourly_pnl_pct': 'Hourly pnl %', 'day_pnl': 'Day pnl', 'day_pnl_pct': 'Day pnl %', 'execution_mode': 'Execution mode'})
    display(trade_tail)


displayHTML('<h2>8. Execution / health / freshness</h2>')
health_rows = [
    ('Dashboard freshness (min)', fmt_minutes(latest_feed.get('dashboard_data_freshness_minutes'))),
    ('Audit lag (min)', fmt_minutes(latest_health.get('audit_lag_minutes'))),
    ('Latest audit ts', fmt_timestamp(latest_feed.get('latest_audit_ts'))),
    ('Last execution action', _as_text(latest_feed.get('last_execution_action'))),
    ('Skip reason', _as_text(latest_feed.get('last_skip_reason'))),
    ('Error reason', _as_text(latest_feed.get('last_error_reason'))),
    ('Skip / 24h', _as_text(latest_feed.get('last_skip_count_24h', 0))),
    ('Error / 24h', _as_text(latest_feed.get('last_error_count_24h', 0))),
    ('Rate-limit / 24h', _as_text(latest_feed.get('last_rate_limit_count_24h', 0))),
]
display(pd.DataFrame(health_rows, columns=['Metric', 'Value']))

if not health_history.empty:
    health_tail = health_history[['rollup_ts', 'audit_lag_minutes', 'last_execution_action', 'last_skip_count_24h', 'last_error_count_24h', 'last_rate_limit_count_24h']].copy()
    health_tail['rollup_ts'] = health_tail['rollup_ts'].dt.strftime('%Y-%m-%d %H:%M')
    display(health_tail.tail(24))
